# 12 — Necessity Hypothesis (Cybersecurity & Integration): Replacing Physical PBX / 必要性假設檢定（資安與整合）

**EN.** This notebook applies **null-hypothesis significance testing** (per the framework at
[yongxi-stat.com/hypothesis-stat](https://www.yongxi-stat.com/hypothesis-stat/)) to the
**cybersecurity + integration** leg of the four-part feasibility frame (financial → NB 10,
technical/lifecycle → NB 11, **cybersecurity & integration → here**). We ask whether the modern
PSTN alternatives are a *materially better and feasibly integrable* security posture than a kept
physical PBX.

> *Do PSTN alternatives deliver a security posture above the physical-PBX baseline, while remaining
> feasible to integrate (bounded complexity / human-time cost)?*

It reuses the **same original data** as notebook 07 (`generate_awesome_list`). It shares **no metric**
with notebook 10 (financial NPV) or notebook 11 (lifecycle obsolescence).

**繁中.** 本筆記本針對四維可行性框架中的**資安＋整合**面向套用虛無假設檢定，沿用與筆記本 07
相同的原始資料（`generate_awesome_list`），與 NB 10、NB 11 不共用任何指標。

## 1. Hypothesis design / 假設設計

**EN.** Two coupled one-sided tests; the **joint H₀** is rejected only if **both** reject:

- **(A) Security necessity** — `security_score` (0–10) scored from each alternative's catalog
  `security` / `pros` text (TLS, SRTP, AES, mTLS, MFA, OAuth, end-to-end).
  - **H₀ₐ:** mean `security_score ≤ 5.0` (neutral physical-PBX baseline). **H₁ₐ:** mean `> 5.0`.
- **(B) Integration feasibility** — `complexity` of the alternatives catalog (mapped 1–7).
  - **H₀_b:** mean complexity `≥ 5` (Medium-High+). **H₁_b:** mean complexity `< 5`.
- **Test:** one-sample, **one-sided t-tests**; report **Cohen's d** and **95% CI**; **α = 0.05**.

**繁中.** 兩個耦合單尾 t 檢定；**聯合 H₀** 僅在兩者皆拒絕時拒絕。
(A) 資安：H₀ₐ：`security_score ≤ 5.0`；H₁ₐ：> 5.0。(B) 整合：H₀_b：複雜度 ≥ 5；H₁_b：< 5。α = 0.05。

In [ ]:
# Run from the repository root (same convention as notebooks 07/10).
import sys
import json
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import stats

ROOT = Path.cwd()
while not (ROOT / "data" / "processed").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.research.tech_researcher import generate_awesome_list, score_security_posture

# --- Original data, identical source to notebook 07 ---
awesome = generate_awesome_list()
print(f"Awesome list loaded: {len(awesome)} PSTN alternatives")

In [ ]:
ALPHA = 0.05
SEC_BASELINE = 5.0       # neutral physical-PBX security baseline (0-10)
COMPLEXITY_FEASIBLE = 5  # mean complexity must be < 5 (below Medium-High) to be integration-feasible

def one_sample_t(sample, popmean, alternative):
    sample = np.asarray(sample, dtype=float)
    sample = sample[~np.isnan(sample)]
    n = sample.size
    res = stats.ttest_1samp(sample, popmean, alternative=alternative)
    sd = sample.std(ddof=1)
    cohens_d = (sample.mean() - popmean) / sd if sd else float("nan")
    se = sd / np.sqrt(n) if n else float("nan")
    tcrit = stats.t.ppf(1 - ALPHA, df=n - 1) if n > 1 else float("nan")
    return {
        "n": int(n), "mean": float(sample.mean()) if n else float("nan"), "popmean": popmean,
        "t_stat": float(res.statistic), "p_value": float(res.pvalue),
        "cohens_d": float(cohens_d),
        "ci_low": float(sample.mean() - tcrit * se), "ci_high": float(sample.mean() + tcrit * se),
        "reject_H0": bool(res.pvalue < ALPHA),
    }

# Security posture from the catalog's free-text security / pros columns.
def row_security_score(row):
    sec = row.get("security", "") or ""
    pros = row.get("pros", "")
    pros_list = pros.split(";") if isinstance(pros, str) else (list(pros) if pros else [])
    return score_security_posture({"tags": [str(sec)], "pros": [str(p) for p in pros_list]})

awesome_sec = awesome.apply(row_security_score, axis=1)

complexity_map = {"Very Low": 1, "Low": 2, "Low-Medium": 3, "Medium": 4,
                  "Medium-High": 5, "High": 6, "Very High": 7}
complexity_num = awesome["complexity"].map(complexity_map)

A = one_sample_t(awesome_sec, SEC_BASELINE, alternative="greater")
B = one_sample_t(complexity_num, COMPLEXITY_FEASIBLE, alternative="less")
joint_reject = bool(A["reject_H0"] and B["reject_H0"])

print("=" * 64)
print("  12  CYBERSECURITY & INTEGRATION — Null-Hypothesis Test")
print("=" * 64)
print("(A) Security necessity   H0: mean security_score <= 5.0   H1: > 5.0")
print(f"    n={A['n']}  mean={A['mean']:.2f}  t={A['t_stat']:.3f}  p={A['p_value']:.4g}")
print(f"    Cohen's d={A['cohens_d']:.3f}  95% CI=[{A['ci_low']:.2f}, {A['ci_high']:.2f}]  reject_H0={A['reject_H0']}")
print("-" * 64)
print("(B) Integration feasible H0: mean complexity >= 5      H1: < 5")
print(f"    n={B['n']}  mean={B['mean']:.2f}  t={B['t_stat']:.3f}  p={B['p_value']:.4g}")
print(f"    Cohen's d={B['cohens_d']:.3f}  95% CI=[{B['ci_low']:.2f}, {B['ci_high']:.2f}]  reject_H0={B['reject_H0']}")
print("=" * 64)
if joint_reject:
    print("  JOINT REJECT H0: replacing the physical PBX is a SECURITY")
    print("  NECESSITY *and* is INTEGRATION-FEASIBLE (bounded complexity).")
else:
    print("  FAIL TO JOINTLY REJECT H0: the cybersecurity+integration case")
    print("  for replacement is NOT established (need both A and B to reject).")
print("=" * 64)

In [ ]:
# Persist verdict CSV (committed back to main by the report workflow)
necessity_security = pd.DataFrame([
    {"dimension": "cybersecurity", "metric": "security_score", "alternative": "greater",
     "popmean": SEC_BASELINE, **{k: A[k] for k in ("n", "mean", "t_stat", "p_value", "cohens_d", "ci_low", "ci_high", "reject_H0")}},
    {"dimension": "integration", "metric": "complexity", "alternative": "less",
     "popmean": COMPLEXITY_FEASIBLE, **{k: B[k] for k in ("n", "mean", "t_stat", "p_value", "cohens_d", "ci_low", "ci_high", "reject_H0")}},
])
necessity_security["alpha"] = ALPHA
necessity_security["joint_reject_H0"] = joint_reject
necessity_security["verdict"] = ("replacement_security_integration_justified" if joint_reject else "not_established")
out_csv = ROOT / "data" / "processed" / "replacement_necessity_security.csv"
necessity_security.to_csv(out_csv, index=False)
print(f"Saved CSV → {out_csv}")
display(necessity_security)

In [ ]:
# ---- Export results to the frontend (same pattern as notebooks 09/10) ----
FRONTEND_DATA = ROOT / "frontend" / "data"
FRONTEND_DATA.mkdir(parents=True, exist_ok=True)

def leg(d, popmean, direction):
    return {
        "n": d["n"], "mean": round(d["mean"], 3), "popmean": popmean, "direction": direction,
        "t_stat": round(d["t_stat"], 3), "p_value": d["p_value"],
        "cohens_d": round(d["cohens_d"], 3),
        "ci_low": round(d["ci_low"], 3), "ci_high": round(d["ci_high"], 3),
        "reject_h0": d["reject_H0"],
    }

payload = {
    "title_en": "Necessity Test — Cybersecurity & Integration",
    "title_zh": "必要性檢定—資安與整合",
    "method_en": "Two coupled one-sided t-tests; the joint H0 is rejected only if BOTH the security and integration legs reject.",
    "method_zh": "兩個耦合單尾 t 檢定；資安與整合兩面向皆拒絕時才拒絕聯合 H0。",
    "alpha": ALPHA,
    "security": leg(A, SEC_BASELINE, "greater"),
    "integration": leg(B, COMPLEXITY_FEASIBLE, "less"),
    "joint_reject_h0": joint_reject,
    "verdict_en": ("Replacement justified on security + integration" if joint_reject
                   else "Security + integration case not established"),
    "verdict_zh": ("資安＋整合面向支持汰換" if joint_reject
                   else "資安＋整合面向尚不足以證明汰換"),
    "sources": [
        {"name": "Hypothesis-testing framework (yongxi-stat)", "url": "https://www.yongxi-stat.com/hypothesis-stat/"},
        {"name": "PSTN alternatives catalog (notebook 07)", "url": "https://github.com/dennislee928/pbx_estimation/blob/main/notebooks/07_tech_alternatives.ipynb"},
    ],
}

out_json = FRONTEND_DATA / "security_integration.json"
out_json.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Frontend JSON → {out_json}")
print(json.dumps(payload, ensure_ascii=False, indent=2)[:600])